# Phần 7: Stacking Ensemble

Ở Phần 4, `train.py` so sánh 8 model đơn rồi chọn 1 model thắng. **Stacking**
đi theo hướng khác: thay vì chọn 1, ta để một model "cấp 2" học cách **trộn
dự đoán** của nhiều model.

**Cách hoạt động (với `StackingRegressor` của sklearn):**
1. Mỗi base model (ví dụ RidgeCV, ExtraTrees, XGBoost, CatBoost) được dự
   đoán *ngoài-fold* trên dữ liệu train: chia train thành K phần, model học
   trên K-1 phần rồi dự đoán phần còn lại. Nhờ vậy dự đoán đưa cho model cấp 2
   là dự đoán trên dòng mà base model **chưa từng thấy** → không rò rỉ.
2. Model cấp 2 (`RidgeCV`) học: "với dự đoán của các base model là a, b, c, d,
   log-giá thật nên là bao nhiêu?" — tức là học trọng số trộn.
3. Cuối cùng các base model được train lại trên toàn bộ dữ liệu train để dùng
   lúc dự đoán.

**Khi nào Stacking có lợi?** Chỉ khi các base model **sai theo những kiểu
khác nhau**. Nếu chúng sai giống nhau thì trộn lại cũng không cải thiện gì
mà còn tốn thời gian. Notebook này kiểm tra đúng điều đó.

> Cấu hình (base model, số fold, `passthrough`, bật/tắt) nằm trong
> `config.yaml`, mục `stacking`. Logic nằm trong `build_stacking()` của
> `src/train.py`.
>
> **Chi phí:** mỗi lần fit, mỗi base model được train `cv_splits + 1` lần.
> Các cell fit bên dưới vì vậy chậm hơn model đơn khá nhiều.

In [ ]:
import sys
sys.path.append('../src')

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from config import CFG
from data_loader import load_processed_data
from train import build_base_models, build_stacking
from evaluate import stacking_weights

df = load_processed_data()
X = df.drop(columns=[CFG['data']['target']])
y = np.log1p(df[CFG['data']['target']])

# Cùng test_size và seed với train.py -> so sánh được với bảng ở Phần 4
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CFG['data']['test_size'], random_state=CFG['seed']
)

def rmsle(model):
    return float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))

## Bước 1: Cấu hình và dựng Stacking

In [ ]:
scfg = CFG['stacking']
print('Base model :', scfg['base_models'])
print('Model cấp 2: RidgeCV, alphas =', scfg['final_alphas'])
print('Số fold sinh dự đoán ngoài-fold:', scfg['cv_splits'])
print('passthrough:', scfg['passthrough'])

models = build_base_models()
stack = build_stacking(models)
print()
print('Các estimator bên trong Stacking:')
for name, est in stack.estimators:
    print(f'  - {name}: {type(est).__name__}')

## Bước 2: Stacking có tốt hơn các base model của nó không?

Train từng base model riêng trên tập train, rồi train Stacking, và chấm tất
cả trên **cùng một tập test giữ riêng**. Đây là phép so sánh công bằng nhất
cho câu hỏi "trộn lại có đáng không".

In [ ]:
names = scfg['base_models']
rows, fitted = [], {}

for n in names:
    m = clone(models[n]).fit(X_train, y_train)
    fitted[n] = m
    rows.append({'model': n, 'RMSLE_test': rmsle(m)})

stack.fit(X_train, y_train)
rows.append({'model': 'Stacking', 'RMSLE_test': rmsle(stack)})

compare = pd.DataFrame(rows).sort_values('RMSLE_test').reset_index(drop=True)
compare

In [ ]:
best_base = compare[compare['model'] != 'Stacking'].iloc[0]
stk = compare.loc[compare['model'] == 'Stacking', 'RMSLE_test'].iloc[0]
gain = (best_base['RMSLE_test'] - stk) / best_base['RMSLE_test'] * 100

print(f"Stacking            : {stk:.4f}")
print(f"Base model tốt nhất : {best_base['model']} = {best_base['RMSLE_test']:.4f}")
print(f"Chênh lệch          : {gain:+.2f}%  (dương = Stacking tốt hơn)")

**Cách đọc kết quả (đọc kỹ trước khi kết luận):**

- Đây là **một** lần chia train/test. Ở Phần 4, độ lệch chuẩn RMSLE giữa các
  fold CV của CatBoost khoảng 0.01 (xem `outputs/reports/cv_comparison.csv`),
  tức khoảng 3% giá trị RMSLE. Một chênh lệch nhỏ hơn mức đó **chưa đủ căn cứ**
  để nói Stacking thực sự tốt hơn — đổi `seed` là có thể đảo chiều.
- Muốn kết luận chắc hơn, so Stacking với model đơn bằng K-fold CV: chạy
  `python src/train.py` rồi xem `outputs/reports/cv_comparison.csv`
  (Stacking đã nằm trong bảng đó).
- Nếu Stacking không hơn rõ rệt, **cứ dùng model đơn** cho gọn và nhanh:
  đặt `stacking.enabled: false` trong `config.yaml`. Với dữ liệu này, Phần 6
  đã cho thấy giới hạn nằm ở dữ liệu (thiếu thông tin vị trí chi tiết) hơn là
  ở model, nên không có gì đảm bảo Stacking sẽ cải thiện đáng kể.

## Bước 3: Model cấp 2 đang trộn như thế nào?

Trọng số của `RidgeCV` cấp 2 cho biết mỗi base model được "tin" bao nhiêu
(trên thang log-giá). Trọng số gần 0 hoặc âm nghĩa là base model đó gần như
không đóng góp thêm gì — ứng viên để bỏ khỏi `stacking.base_models` cho đỡ
tốn thời gian train.

In [ ]:
weights = stacking_weights(stack, X.columns)
print('Hệ số chặn (intercept):', round(float(stack.final_estimator_.intercept_), 4))
print('Tổng trọng số base model:', round(float(weights[[n for n, _ in stack.estimators]].sum()), 3))

top = weights.reindex(weights.abs().sort_values(ascending=False).index).head(15)
plt.figure(figsize=(7, 4))
top.sort_values().plot(kind='barh', color='teal')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Trọng số trộn của model cấp 2')
plt.tight_layout()
plt.show()
weights.round(3)

## Bước 4: Các base model có sai khác nhau không?

Đây là lý do Stacking có thể (hoặc không thể) hiệu quả. Ta tính **residual**
(log thực tế − log dự đoán) của từng base model trên tập test rồi xem tương
quan giữa chúng:

- Tương quan **gần 1**: hai model mắc gần như cùng lỗi trên cùng những căn nhà
  → trộn lại không thêm được thông tin mới.
- Tương quan **thấp hơn rõ rệt**: chúng sai ở những chỗ khác nhau → model cấp
  2 có thể bù trừ lỗi cho nhau.

In [ ]:
resid = pd.DataFrame({n: y_test - fitted[n].predict(X_test) for n in names})
corr = resid.corr()

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='viridis', vmin=corr.values.min() - 0.02, vmax=1)
plt.title('Tương quan residual giữa các base model')
plt.tight_layout()
plt.show()

pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(names) for b in names[i + 1:]]
a, b, c = max(pairs, key=lambda p: p[2])
print(f'Cặp giống nhau nhất: {a} & {b} (tương quan residual = {c:.2f})')
a, b, c = min(pairs, key=lambda p: p[2])
print(f'Cặp khác nhau nhất : {a} & {b} (tương quan residual = {c:.2f})')

## Bước 5 (tùy chọn, chậm): Bỏ từng base model ra thì sao?

Fit lại Stacking **một lần cho mỗi base model bị bỏ** — với XGBoost/CatBoost
thật, việc này mất khá lâu nên mặc định tắt. Đổi `RUN_ABLATION = True` để
chạy. Kết quả cho biết base model nào thực sự cần thiết: nếu bỏ đi mà RMSLE
gần như không đổi, có thể loại nó khỏi `config.yaml`.

In [ ]:
RUN_ABLATION = False   # đổi thành True để chạy (chậm)

if RUN_ABLATION:
    if len(names) < 3:
        print('Cần ít nhất 3 base model để bỏ bớt 1 mà vẫn còn >= 2.')
    else:
        rows = [{'bỏ base model': '(không bỏ gì)', 'RMSLE_test': stk}]
        for drop in names:
            cfg2 = copy.deepcopy(CFG)
            cfg2['stacking']['base_models'] = [n for n in names if n != drop]
            s2 = build_stacking(build_base_models(cfg2), cfg2).fit(X_train, y_train)
            rows.append({'bỏ base model': drop, 'RMSLE_test': rmsle(s2)})
        ablation = pd.DataFrame(rows)
        display(ablation) if 'display' in globals() else print(ablation)
else:
    print('Bỏ qua ablation (RUN_ABLATION = False).')

## Kết luận và bước tiếp theo

- `python src/train.py` sẽ tự đưa Stacking vào cuộc so sánh 9 model và lưu
  model thắng vào `models/best_model.pkl` (kèm `model_meta.json`). Notebook này
  **không** ghi đè model đã lưu.
- Sau khi `train.py` chạy xong, hãy cập nhật bảng kết quả trong `README.md`
  bằng số thật, kể cả khi Stacking không thắng — đó cũng là một kết quả đáng
  ghi lại.
- Nếu Stacking là model tốt nhất, `python src/evaluate.py` sẽ in trọng số trộn
  thay cho feature importance, còn `python src/predict.py` dùng nó như mọi
  model khác (không cần đổi gì).